# Fine-tune Qwen2.5-3B-Instruct on a RAFT dataset (QLoRA)

Fine-tunes a 3B model with **QLoRA** (4-bit base model + LoRA adapter) on a synthetic **RAFT** dataset built from the FastAPI documentation: each example is 4 retrieved documents + a question, and the target answer is short reasoning with verbatim quotes followed by `<ANSWER>:`. Part of the *fastapi-raft* project (see the repository README).

In [7]:
!pip install -q -U peft bitsandbytes accelerate

## 1. Settings

In [8]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   

import json, random
import torch

# base model
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"    

MAX_LEN = 2048                              
EPOCHS = 5                                  
LEARNING_RATE = 1e-4
SEED = 42
OUTPUT_DIR = "/kaggle/working/raft-lora"

# Path of the training file on Kaggle.
DATA_FILE = "/kaggle/input/datasets/oualidlaib/new-dt/raft_train_set_final.jsonl"
print("Using data file:", DATA_FILE)

Using data file: /kaggle/input/datasets/oualidlaib/new-dt/raft_train_set_final.jsonl


## 2. Prepare the data

Each example is turned into tokens. The **labels** of the question part (documents + question) are set to `-100`, so the model is trained **only on the answer** (reasoning + quotes + `<ANSWER>:`), not on the documents. Examples longer than `MAX_LEN` tokens are dropped, not truncated, because cutting would remove the answer. The split is 90% train / 10% validation, by example.

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

rows = [json.loads(line) for line in open(DATA_FILE, encoding="utf-8")]
print(len(rows), "examples in the file")


def encode(example):

    # [user message, assistant message]
    messages = example["messages"]   

    prompt_text = tokenizer.apply_chat_template(messages[:1], tokenize=False, add_generation_prompt=True)

    # Full text: question + answer
    full_text = tokenizer.apply_chat_template(messages, tokenize=False)
    assert full_text.startswith(prompt_text), "chat template does not split cleanly"

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]   

    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}


# Encode everything and drop long examples
pairs = [(row, encode(row)) for row in rows]
lengths = [len(enc["input_ids"]) for _, enc in pairs]
print("token length: min", min(lengths), "| max", max(lengths), "| average", sum(lengths) // len(lengths))

pairs = [(row, enc) for row, enc in pairs if len(enc["input_ids"]) <= MAX_LEN]
print(len(pairs), "examples kept (<= MAX_LEN tokens)")

# 90% train / 10% validation
random.Random(SEED).shuffle(pairs)
n_val = max(1, int(len(pairs) * 0.1))
val_pairs, train_pairs = pairs[:n_val], pairs[n_val:]

train_data = [enc for _, enc in train_pairs]
val_data = [enc for _, enc in val_pairs]
val_rows = [row for row, _ in val_pairs]
print("train:", len(train_data), "| validation:", len(val_data))

264 examples in the file
token length: min 430 | max 2865 | average 963
261 examples kept (<= MAX_LEN tokens)
train: 235 | validation: 26


## 3. Load the model in 4-bit and add LoRA

In [10]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,   
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


## 4. Train

In [11]:
# Read the Weights & Biases key from Kaggle Secrets
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WANDB_API_KEY")

In [12]:
import os, wandb
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq, EarlyStoppingCallback

os.environ["WANDB_PROJECT"] = "fastapi_tune-v2"
wandb.login(secret_value_0)

GRAD_ACCUM = 4                                        
steps_per_epoch = len(train_data) // GRAD_ACCUM       
EVAL_STEPS = max(10, steps_per_epoch // 2)            

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,                     
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,            
    lr_scheduler_type="cosine",
    warmup_steps=10,
    weight_decay=0.01,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",

    # log the training loss every 5 steps
    logging_steps=5,                        

    # evaluate every EVAL_STEPS steps
    eval_strategy="steps",                  
    eval_steps=EVAL_STEPS,
    save_strategy="steps",                  
    save_steps=EVAL_STEPS,

    # keep the best and the latest checkpoint
    save_total_limit=2,                     

    # restore the checkpoint with the lowest eval loss
    load_best_model_at_end=True,           
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="wandb",
    run_name="raft-lora-v5",                
    seed=SEED,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100),

    # stop after 4 evaluations without improvement
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],   
)
model.config.use_cache = False

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: laiboualid2003 (laiboualid2003-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [13]:
trainer.train()
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best eval loss:", trainer.state.best_metric)

Step,Training Loss,Validation Loss
29,0.607193,0.635057
58,0.528820,0.569835
87,0.380406,0.568774
116,0.551383,0.560281
145,0.293569,0.641760
174,0.388162,0.636453
203,0.157001,0.815523
232,0.123799,0.781669


Best checkpoint: /kaggle/working/raft-lora/checkpoint-116
Best eval loss: 0.5602813363075256


## 5. Save the best adapter

Saves the LoRA adapter and tokenizer to `/kaggle/working/raft-lora/best`.

In [14]:
best_dir = OUTPUT_DIR + "/best"

# LoRA adapter weights
model.save_pretrained(best_dir)         
tokenizer.save_pretrained(best_dir)

wandb.run.summary["best_eval_loss"] = trainer.state.best_metric
wandb.finish()

eval/loss,▃▁▁▁▃▃█▇
eval/runtime,▁▅▄▆▁▅█▆
eval/samples_per_second,█▄▅▃█▄▁▃
eval/steps_per_second,█▄▅▃█▄▁▃
train/epoch,▁▁▁▁▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇████
train/global_step,▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
train/grad_norm,█▃▃▂▃▂▂▂▂▁▂▂▁▁▃▁▂▁▁▃▂▂▂▃▄▄▂▃▃▃▂▂▂▃▄▅▂▂▄▂
train/learning_rate,▃▇████████▇▇▇▇▇▇▆▆▆▆▅▅▅▅▅▄▄▄▃▃▃▃▃▂▂▂▂▁▁▁
train/loss,█▅▅▄▃▃▂▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
best_eval_loss,0.56028
eval/loss,0.78167
